# Advanced Problems with Solutions: Sending Data to Python Generators

This notebook develops the ideas behind **PEP 342-style generator coroutines**: a suspended generator can receive values through `send()`, receive exceptions through `throw()`, and be finalized through `close()`.

The notebook is deliberately problem-driven. Each section contains:

1. a precise problem,
2. design constraints and edge cases,
3. a complete solution,
4. executable checks,
5. best-practice notes.

> **Target level:** intermediate-to-advanced Python.  
> **Recommended Python:** 3.10 or newer.  
> **Scope note:** these are synchronous generator coroutines, not `async def` coroutines.

## Learning objectives

By the end, you should be able to:

- explain exactly where a value sent by `send(x)` appears inside a generator;
- distinguish `GEN_CREATED`, `GEN_SUSPENDED`, and `GEN_CLOSED` states;
- design explicit message protocols instead of sending ambiguous raw values;
- use `throw()` for control signals and `close()` for deterministic cleanup;
- implement numerically stable streaming algorithms;
- compose coroutines with `yield from` and capture subgenerator return values;
- test coroutine state transitions, invalid inputs, termination, and cleanup;
- build a non-trivial streaming telemetry processor.

## Best-practice checklist

- **Prime deliberately.** A newly created generator cannot receive a non-`None` value. Use `next(g)` or `g.send(None)` once.
- **Document the protocol.** State what may be sent and what each call returns.
- **Prefer typed messages.** Enums and dataclasses are clearer than magic strings and tuples.
- **Do not leak mutable internal state.** Yield immutable snapshots or defensive copies.
- **Handle expected bad input inside the coroutine.** An uncaught exception closes the generator.
- **Use `try/finally` for cleanup.** `close()` injects `GeneratorExit` at the suspension point.
- **Avoid yielding after `GeneratorExit`.** Doing so raises `RuntimeError`.
- **Test lifecycle transitions.** Functional output alone does not prove the protocol is correct.
- **Know when not to use this pattern.** For general application code, classes, iterators, queues, or `asyncio` are often clearer.

In [1]:
from __future__ import annotations

from collections import defaultdict, deque
from collections.abc import Generator, Iterable, Iterator
from contextlib import contextmanager
from dataclasses import dataclass, field, replace
from enum import Enum, auto
from functools import wraps
from inspect import getgeneratorstate
from math import isfinite, sqrt
from typing import Any, Callable, Optional, TypeVar

T = TypeVar("T")

# Problem 1 — Model the generator lifecycle exactly

Create a generator that remembers the most recently sent value and yields that value back on the next suspension.

### Requirements

- Before priming, the generator must be in `GEN_CREATED`.
- Sending a non-`None` value before priming must fail with `TypeError`.
- After priming, it must be in `GEN_SUSPENDED`.
- `send("alpha")` must return `"alpha"`.
- After `close()`, it must be in `GEN_CLOSED`.

## Solution 1

In [2]:
def remember_last() -> Generator[Optional[str], Optional[str], None]:
    """Yield the last value received; initially yield None."""
    last: Optional[str] = None
    while True:
        last = yield last


g = remember_last()
print("created:", getgeneratorstate(g))

try:
    g.send("too early")
except TypeError as exc:
    print("expected priming error:", exc)

print("first yield:", next(g))
print("suspended:", getgeneratorstate(g))
print("send alpha ->", g.send("alpha"))
print("send beta  ->", g.send("beta"))

g.close()
print("closed:", getgeneratorstate(g))

created: GEN_CREATED
expected priming error: can't send non-None value to a just-started generator
first yield: None
suspended: GEN_SUSPENDED
send alpha -> alpha
send beta  -> beta
closed: GEN_CLOSED


In [3]:
g = remember_last()
assert getgeneratorstate(g) == "GEN_CREATED"
assert next(g) is None
assert getgeneratorstate(g) == "GEN_SUSPENDED"
assert g.send("alpha") == "alpha"
assert g.send("beta") == "beta"
g.close()
assert getgeneratorstate(g) == "GEN_CLOSED"
print("Problem 1 checks passed.")

Problem 1 checks passed.


### Reasoning

For `last = yield last`, execution happens in two phases:

1. the current `last` is yielded and the generator suspends;
2. a later `send(value)` makes the `yield` expression evaluate to `value`, which is then assigned to `last`.

`next(g)` is equivalent to `g.send(None)` for a suspended generator.

# Problem 2 — Design a command-driven running total

Build a coroutine that accepts both numeric data and explicit commands.

### Protocol

The caller may send:

- a finite `int` or `float`: add it;
- `TotalCommand.RESET`: reset total and count;
- `TotalCommand.SNAPSHOT`: return the current snapshot without changing it;
- `TotalCommand.STOP`: terminate and return the final snapshot through `StopIteration.value`.

Every non-terminal `send()` returns an immutable `TotalSnapshot`.

## Solution 2

In [4]:
class TotalCommand(Enum):
    RESET = auto()
    SNAPSHOT = auto()
    STOP = auto()


@dataclass(frozen=True)
class TotalSnapshot:
    total: float
    count: int

    @property
    def average(self) -> Optional[float]:
        return None if self.count == 0 else self.total / self.count


def running_total() -> Generator[TotalSnapshot, float | TotalCommand, TotalSnapshot]:
    total = 0.0
    count = 0
    message = yield TotalSnapshot(total, count)

    while True:
        if message is TotalCommand.RESET:
            total = 0.0
            count = 0
        elif message is TotalCommand.SNAPSHOT:
            pass
        elif message is TotalCommand.STOP:
            return TotalSnapshot(total, count)
        elif isinstance(message, (int, float)) and not isinstance(message, bool):
            value = float(message)
            if not isfinite(value):
                raise ValueError("Only finite numbers are accepted")
            total += value
            count += 1
        else:
            raise TypeError(f"Unsupported message: {message!r}")

        message = yield TotalSnapshot(total, count)

In [5]:
totals = running_total()
print(next(totals))
print(totals.send(10))
print(totals.send(5.5))
print(totals.send(TotalCommand.SNAPSHOT))
print(totals.send(TotalCommand.RESET))
print(totals.send(7))

try:
    totals.send(TotalCommand.STOP)
except StopIteration as exc:
    final_total = exc.value
    print("returned final snapshot:", final_total)

TotalSnapshot(total=0.0, count=0)
TotalSnapshot(total=10.0, count=1)
TotalSnapshot(total=15.5, count=2)
TotalSnapshot(total=15.5, count=2)
TotalSnapshot(total=0.0, count=0)
TotalSnapshot(total=7.0, count=1)
returned final snapshot: TotalSnapshot(total=7.0, count=1)


In [6]:
totals = running_total()
assert next(totals) == TotalSnapshot(0.0, 0)
assert totals.send(10) == TotalSnapshot(10.0, 1)
assert totals.send(5) == TotalSnapshot(15.0, 2)
assert totals.send(TotalCommand.RESET) == TotalSnapshot(0.0, 0)
assert totals.send(3).average == 3.0
try:
    totals.send(TotalCommand.STOP)
except StopIteration as exc:
    assert exc.value == TotalSnapshot(3.0, 1)
else:
    raise AssertionError("STOP should terminate the coroutine")
print("Problem 2 checks passed.")

Problem 2 checks passed.


### Best-practice note

This solution intentionally raises on unsupported messages, so misuse closes the coroutine. In a long-running service, returning an error response may be preferable. Problem 4 demonstrates that more fault-tolerant style.

# Problem 3 — Implement a safe auto-priming decorator

Repeatedly writing `g = func(); next(g)` is noisy. Implement a decorator that primes a generator coroutine automatically.

### Requirements

- Preserve the wrapped function's metadata.
- Return the primed generator object.
- Fail clearly when the wrapped generator terminates before its first `yield`.
- Demonstrate the decorator with a sink that appends received values to a list.

## Solution 3

In [7]:
G = TypeVar("G", bound=Generator[Any, Any, Any])


def autoprime(factory: Callable[..., G]) -> Callable[..., G]:
    """Wrap a generator factory and advance each new generator to first yield."""
    @wraps(factory)
    def wrapper(*args: Any, **kwargs: Any) -> G:
        gen = factory(*args, **kwargs)
        try:
            next(gen)
        except StopIteration as exc:
            raise RuntimeError(
                f"{factory.__name__} terminated before reaching its first yield"
            ) from exc
        return gen

    return wrapper


@autoprime
def list_sink(target: list[Any]) -> Generator[None, Any, None]:
    while True:
        item = yield
        target.append(item)

In [8]:
received: list[Any] = []
sink = list_sink(received)
print("state after construction:", getgeneratorstate(sink))
sink.send("A")
sink.send("B")
sink.send({"id": 3})
print(received)
sink.close()

state after construction: GEN_SUSPENDED
['A', 'B', {'id': 3}]


In [9]:
received = []
sink = list_sink(received)
assert getgeneratorstate(sink) == "GEN_SUSPENDED"
assert sink.send(1) is None
assert sink.send(2) is None
assert received == [1, 2]
sink.close()
print("Problem 3 checks passed.")

Problem 3 checks passed.


### Trade-off

Auto-priming is convenient but hides a state transition. Use it only when every caller should receive an already-primed coroutine. Public APIs should state this explicitly.

# Problem 4 — Numerically stable streaming statistics

Implement a fault-tolerant coroutine that computes count, mean, sample variance, and standard deviation in one pass using **Welford's algorithm**.

### Requirements

- Accept finite real numbers.
- Reject booleans, `NaN`, and infinities without closing the coroutine.
- Accept `StatsCommand.RESET` and `StatsCommand.SNAPSHOT`.
- Return immutable snapshots.
- Avoid the unstable formula `E[x²] - E[x]²`.

## Solution 4

In [10]:
class StatsCommand(Enum):
    RESET = auto()
    SNAPSHOT = auto()


@dataclass(frozen=True)
class StatsSnapshot:
    count: int
    mean: float
    m2: float

    @property
    def sample_variance(self) -> Optional[float]:
        return None if self.count < 2 else self.m2 / (self.count - 1)

    @property
    def population_variance(self) -> Optional[float]:
        return None if self.count == 0 else self.m2 / self.count

    @property
    def sample_stddev(self) -> Optional[float]:
        variance = self.sample_variance
        return None if variance is None else sqrt(variance)


@dataclass(frozen=True)
class StatsResponse:
    stats: StatsSnapshot
    error: Optional[str] = None


def streaming_stats() -> Generator[StatsResponse, float | StatsCommand, None]:
    count = 0
    mean = 0.0
    m2 = 0.0

    def snapshot() -> StatsSnapshot:
        return StatsSnapshot(count=count, mean=mean, m2=m2)

    message = yield StatsResponse(snapshot())

    while True:
        error: Optional[str] = None

        if message is StatsCommand.RESET:
            count, mean, m2 = 0, 0.0, 0.0
        elif message is StatsCommand.SNAPSHOT:
            pass
        elif isinstance(message, (int, float)) and not isinstance(message, bool):
            value = float(message)
            if not isfinite(value):
                error = f"non-finite value rejected: {value!r}"
            else:
                count += 1
                delta = value - mean
                mean += delta / count
                delta2 = value - mean
                m2 += delta * delta2
        else:
            error = f"unsupported message rejected: {message!r}"

        message = yield StatsResponse(snapshot(), error)

In [11]:
stats = streaming_stats()
print(next(stats))
for value in [1, 2, 3, 4]:
    print(stats.send(value))

print("bad input:", stats.send(float("nan")))
print("still alive:", getgeneratorstate(stats))
print("reset:", stats.send(StatsCommand.RESET))
stats.close()

StatsResponse(stats=StatsSnapshot(count=0, mean=0.0, m2=0.0), error=None)
StatsResponse(stats=StatsSnapshot(count=1, mean=1.0, m2=0.0), error=None)
StatsResponse(stats=StatsSnapshot(count=2, mean=1.5, m2=0.5), error=None)
StatsResponse(stats=StatsSnapshot(count=3, mean=2.0, m2=2.0), error=None)
StatsResponse(stats=StatsSnapshot(count=4, mean=2.5, m2=5.0), error=None)
bad input: StatsResponse(stats=StatsSnapshot(count=4, mean=2.5, m2=5.0), error='non-finite value rejected: nan')
still alive: GEN_SUSPENDED
reset: StatsResponse(stats=StatsSnapshot(count=0, mean=0.0, m2=0.0), error=None)


In [12]:
stats = streaming_stats()
next(stats)
responses = [stats.send(x) for x in [1, 2, 3, 4]]
final = responses[-1].stats
assert final.count == 4
assert abs(final.mean - 2.5) < 1e-12
assert abs(final.sample_variance - (5 / 3)) < 1e-12
rejected = stats.send(float("inf"))
assert rejected.error is not None
assert rejected.stats == final
assert getgeneratorstate(stats) == "GEN_SUSPENDED"
stats.close()
print("Problem 4 checks passed.")

Problem 4 checks passed.


### Why Welford's algorithm?

It updates the mean and sum of squared deviations incrementally, uses constant memory, and is much less vulnerable to catastrophic cancellation for large values with small variance.

# Problem 5 — Use `throw()` as an out-of-band control channel

Create a summing coroutine with these properties:

- normal numeric messages add to the total;
- `g.throw(SkipCurrent)` skips an update and immediately yields the unchanged total;
- `g.throw(ResetTotal)` resets the total and yields zero;
- `close()` executes cleanup code exactly once.

## Solution 5

In [13]:
class SkipCurrent(Exception):
    """Signal: ignore the current update opportunity."""


class ResetTotal(Exception):
    """Signal: reset accumulated state."""


def exception_controlled_sum(cleanup_log: list[str]) -> Generator[float, float, None]:
    total = 0.0
    try:
        while True:
            try:
                value = yield total
            except SkipCurrent:
                continue
            except ResetTotal:
                total = 0.0
                continue

            if isinstance(value, bool) or not isinstance(value, (int, float)):
                raise TypeError("numeric values required")
            total += float(value)
    finally:
        cleanup_log.append("sum coroutine cleaned up")

In [14]:
cleanup_log: list[str] = []
summer = exception_controlled_sum(cleanup_log)
print(next(summer))
print(summer.send(10))
print("skip ->", summer.throw(SkipCurrent))
print(summer.send(5))
print("reset ->", summer.throw(ResetTotal))
print(summer.send(2.5))
summer.close()
print(cleanup_log)

0.0
10.0
skip -> 10.0
15.0
reset -> 0.0
2.5
['sum coroutine cleaned up']


In [15]:
cleanup_log = []
summer = exception_controlled_sum(cleanup_log)
assert next(summer) == 0.0
assert summer.send(4) == 4.0
assert summer.throw(SkipCurrent) == 4.0
assert summer.throw(ResetTotal) == 0.0
assert summer.send(3) == 3.0
summer.close()
summer.close()  # closing an already closed generator is harmless
assert cleanup_log == ["sum coroutine cleaned up"]
assert getgeneratorstate(summer) == "GEN_CLOSED"
print("Problem 5 checks passed.")

Problem 5 checks passed.


### Best-practice note

`throw()` can make protocols harder to discover because control messages are no longer ordinary values. Reserve it for exceptional or truly out-of-band signals, and document every accepted exception type.

# Problem 6 — Build a bounded batching coroutine

Implement a coroutine that groups incoming items into fixed-size batches.

### Protocol

- Sending a normal item returns `None` until a full batch is ready.
- Sending `BatchCommand.FLUSH` returns the partial batch, or an empty tuple when nothing is buffered.
- Sending `BatchCommand.STOP` terminates and returns the final partial batch via `StopIteration.value`.
- A batch must be returned as an immutable tuple so the caller cannot mutate internal state.

## Solution 6

In [16]:
class BatchCommand(Enum):
    FLUSH = auto()
    STOP = auto()


def batcher(size: int) -> Generator[Optional[tuple[Any, ...]], Any | BatchCommand, tuple[Any, ...]]:
    if isinstance(size, bool) or not isinstance(size, int) or size <= 0:
        raise ValueError("size must be a positive integer")

    buffer: list[Any] = []
    message = yield None

    while True:
        emitted: Optional[tuple[Any, ...]] = None

        if message is BatchCommand.FLUSH:
            emitted = tuple(buffer)
            buffer.clear()
        elif message is BatchCommand.STOP:
            return tuple(buffer)
        else:
            buffer.append(message)
            if len(buffer) == size:
                emitted = tuple(buffer)
                buffer.clear()

        message = yield emitted

In [17]:
b = batcher(3)
print(next(b))
for item in ["a", "b", "c", "d"]:
    print(f"send {item!r} ->", b.send(item))
print("flush ->", b.send(BatchCommand.FLUSH))

try:
    b.send(BatchCommand.STOP)
except StopIteration as exc:
    print("remaining at stop:", exc.value)

None
send 'a' -> None
send 'b' -> None
send 'c' -> ('a', 'b', 'c')
send 'd' -> None
flush -> ('d',)
remaining at stop: ()


In [18]:
b = batcher(2)
assert next(b) is None
assert b.send(1) is None
assert b.send(2) == (1, 2)
assert b.send(3) is None
assert b.send(BatchCommand.FLUSH) == (3,)
assert b.send(BatchCommand.FLUSH) == ()
try:
    b.send(BatchCommand.STOP)
except StopIteration as exc:
    assert exc.value == ()
else:
    raise AssertionError("STOP must terminate")
print("Problem 6 checks passed.")

Problem 6 checks passed.


# Problem 7 — Route events to multiple coroutine sinks

Build a small publish/subscribe router.

### Requirements

- Events are immutable and contain a `topic` and `payload`.
- Multiple sinks may subscribe to the same topic.
- A default sink receives events whose topic has no subscribers.
- Each send returns the number of sinks that received the event.
- The router does **not** own the sinks, so closing the router must not close them automatically.

## Solution 7

In [19]:
@dataclass(frozen=True)
class Event:
    topic: str
    payload: Any


@autoprime
def event_sink(target: list[Event]) -> Generator[None, Event, None]:
    while True:
        event = yield
        target.append(event)


def event_router(
    routes: dict[str, tuple[Generator[None, Event, None], ...]],
    default: Optional[Generator[None, Event, None]] = None,
) -> Generator[int, Event, None]:
    delivered = 0
    event = yield delivered

    while True:
        sinks = routes.get(event.topic)
        if sinks:
            for sink in sinks:
                sink.send(event)
            delivered = len(sinks)
        elif default is not None:
            default.send(event)
            delivered = 1
        else:
            delivered = 0

        event = yield delivered

In [20]:
audit_events: list[Event] = []
metric_events: list[Event] = []
default_events: list[Event] = []

audit_sink = event_sink(audit_events)
metric_sink = event_sink(metric_events)
default_sink = event_sink(default_events)

router = event_router(
    routes={
        "audit": (audit_sink,),
        "metric": (metric_sink, audit_sink),
    },
    default=default_sink,
)

print(next(router))
print(router.send(Event("audit", {"user": "Ada"})))
print(router.send(Event("metric", 98.7)))
print(router.send(Event("unknown", "fallback")))
print("audit:", audit_events)
print("metric:", metric_events)
print("default:", default_events)

router.close()
for sink in (audit_sink, metric_sink, default_sink):
    sink.close()

0
1
2
1
audit: [Event(topic='audit', payload={'user': 'Ada'}), Event(topic='metric', payload=98.7)]
metric: [Event(topic='metric', payload=98.7)]
default: [Event(topic='unknown', payload='fallback')]


In [21]:
a, m, d = [], [], []
sa, sm, sd = event_sink(a), event_sink(m), event_sink(d)
r = event_router({"x": (sa, sm)}, sd)
assert next(r) == 0
assert r.send(Event("x", 1)) == 2
assert r.send(Event("other", 2)) == 1
assert a == [Event("x", 1)]
assert m == [Event("x", 1)]
assert d == [Event("other", 2)]
r.close()
assert getgeneratorstate(sa) == "GEN_SUSPENDED"  # router does not own it
for sink in (sa, sm, sd):
    sink.close()
print("Problem 7 checks passed.")

Problem 7 checks passed.


### Ownership rule

Resource ownership should be explicit. A router that receives externally created sinks should not silently close them. An alternative API could accept sink factories and then clearly own their lifecycle.

# Problem 8 — Parse a streaming record protocol with a state machine

Incoming text lines follow this protocol:

```text
BEGIN
key=value
key=value
END
```

Build a coroutine parser that returns a completed dictionary only after `END`.

### Requirements

- Ignore blank lines outside a record.
- Reject nested `BEGIN` without closing the parser.
- Reject malformed field lines without closing the parser.
- Reject duplicate keys.
- `ABORT` discards the current record.
- Every send returns a `ParseResponse` containing either a record, an error, or neither.

## Solution 8

In [22]:
@dataclass(frozen=True)
class ParseResponse:
    record: Optional[dict[str, str]] = None
    error: Optional[str] = None
    in_record: bool = False


def record_parser() -> Generator[ParseResponse, str, None]:
    current: Optional[dict[str, str]] = None
    line = yield ParseResponse(in_record=False)

    while True:
        record: Optional[dict[str, str]] = None
        error: Optional[str] = None
        token = line.strip()

        if token == "BEGIN":
            if current is not None:
                error = "nested BEGIN is not allowed"
            else:
                current = {}
        elif token == "END":
            if current is None:
                error = "END received outside a record"
            else:
                record = dict(current)
                current = None
        elif token == "ABORT":
            current = None
        elif not token:
            if current is not None:
                error = "blank lines are not allowed inside a record"
        elif current is None:
            error = "field received outside a record"
        elif "=" not in token:
            error = "field must use key=value format"
        else:
            key, value = (part.strip() for part in token.split("=", 1))
            if not key:
                error = "field key cannot be empty"
            elif key in current:
                error = f"duplicate key: {key}"
            else:
                current[key] = value

        line = yield ParseResponse(
            record=record,
            error=error,
            in_record=current is not None,
        )

In [23]:
parser = record_parser()
print(next(parser))
for line in [
    "BEGIN",
    "name = Ada",
    "language = Python",
    "language = duplicate",
    "END",
    "orphan=value",
]:
    print(f"{line!r} ->", parser.send(line))
parser.close()

ParseResponse(record=None, error=None, in_record=False)
'BEGIN' -> ParseResponse(record=None, error=None, in_record=True)
'name = Ada' -> ParseResponse(record=None, error=None, in_record=True)
'language = Python' -> ParseResponse(record=None, error=None, in_record=True)
'language = duplicate' -> ParseResponse(record=None, error='duplicate key: language', in_record=True)
'END' -> ParseResponse(record={'name': 'Ada', 'language': 'Python'}, error=None, in_record=False)
'orphan=value' -> ParseResponse(record=None, error='field received outside a record', in_record=False)


In [24]:
parser = record_parser()
next(parser)
assert parser.send("BEGIN").in_record
assert parser.send("id=42").error is None
duplicate = parser.send("id=43")
assert duplicate.error == "duplicate key: id"
completed = parser.send("END")
assert completed.record == {"id": "42"}
assert not completed.in_record
assert parser.send("BEGIN").in_record
assert not parser.send("ABORT").in_record
parser.close()
print("Problem 8 checks passed.")

Problem 8 checks passed.


# Problem 9 — Delegate with `yield from` and capture return values

Create a subgenerator that receives numbers for one group. Sending `None` ends the group and **returns** its average. A delegating coroutine repeatedly runs groups and collects all returned averages.

### Outer protocol

1. Prime the outer coroutine.
2. Send numbers; each send returns `None` while the group is open.
3. Send `None`; the call returns the completed average.
4. Send `GroupCommand.NEXT` to start another group.
5. After a completed group, send `GroupCommand.STOP` to terminate and retrieve all averages from `StopIteration.value`.

## Solution 9

In [25]:
class GroupCommand(Enum):
    NEXT = auto()
    STOP = auto()


def average_group() -> Generator[None, Optional[float], Optional[float]]:
    total = 0.0
    count = 0

    while True:
        value = yield
        if value is None:
            return None if count == 0 else total / count
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError("group values must be numeric or None")
        total += float(value)
        count += 1


def grouped_averages() -> Generator[Optional[float], Optional[float] | GroupCommand, list[Optional[float]]]:
    completed: list[Optional[float]] = []

    while True:
        average = yield from average_group()
        completed.append(average)
        command = yield average

        if command is GroupCommand.STOP:
            return completed
        if command is not GroupCommand.NEXT:
            raise ValueError("send GroupCommand.NEXT or GroupCommand.STOP after a group")

In [26]:
groups = grouped_averages()
print("prime ->", next(groups))
print("10 ->", groups.send(10))
print("20 ->", groups.send(20))
print("end group ->", groups.send(None))
print("next group ->", groups.send(GroupCommand.NEXT))
print("4 ->", groups.send(4))
print("6 ->", groups.send(6))
print("end group ->", groups.send(None))

try:
    groups.send(GroupCommand.STOP)
except StopIteration as exc:
    print("all averages:", exc.value)

prime -> None
10 -> None
20 -> None
end group -> 15.0
next group -> None
4 -> None
6 -> None
end group -> 5.0
all averages: [15.0, 5.0]


In [27]:
groups = grouped_averages()
assert next(groups) is None
assert groups.send(2) is None
assert groups.send(4) is None
assert groups.send(None) == 3.0
assert groups.send(GroupCommand.NEXT) is None
assert groups.send(None) is None  # empty group average
try:
    groups.send(GroupCommand.STOP)
except StopIteration as exc:
    assert exc.value == [3.0, None]
else:
    raise AssertionError("STOP must terminate")
print("Problem 9 checks passed.")

Problem 9 checks passed.


### Key point

`yield from subgenerator` forwards `send()`, `throw()`, and `close()` interactions to the subgenerator. When the subgenerator returns, its return value becomes the value of the `yield from` expression.

# Problem 10 — Implement a bidirectional bounded buffer

A bounded buffer demonstrates a request/response protocol and a form of backpressure.

### Messages

- `Put(value)` adds an item if capacity remains.
- `Get()` removes the oldest item.
- `BufferCommand.STATUS` reports occupancy.
- `BufferCommand.STOP` terminates and returns remaining items.

Each operation returns a structured `BufferResponse` instead of raising for ordinary full/empty conditions.

## Solution 10

In [28]:
@dataclass(frozen=True)
class Put:
    value: Any


@dataclass(frozen=True)
class Get:
    pass


class BufferCommand(Enum):
    STATUS = auto()
    STOP = auto()


@dataclass(frozen=True)
class BufferResponse:
    ok: bool
    status: str
    value: Any = None
    size: int = 0
    capacity: int = 0


def bounded_buffer(capacity: int) -> Generator[BufferResponse, Put | Get | BufferCommand, tuple[Any, ...]]:
    if isinstance(capacity, bool) or not isinstance(capacity, int) or capacity <= 0:
        raise ValueError("capacity must be a positive integer")

    queue: deque[Any] = deque()

    def response(ok: bool, status: str, value: Any = None) -> BufferResponse:
        return BufferResponse(ok, status, value, len(queue), capacity)

    message = yield response(True, "ready")

    while True:
        if isinstance(message, Put):
            if len(queue) >= capacity:
                result = response(False, "full")
            else:
                queue.append(message.value)
                result = response(True, "stored")
        elif isinstance(message, Get):
            if queue:
                result = response(True, "retrieved", queue.popleft())
            else:
                result = response(False, "empty")
        elif message is BufferCommand.STATUS:
            result = response(True, "status")
        elif message is BufferCommand.STOP:
            return tuple(queue)
        else:
            result = response(False, f"unsupported message: {message!r}")

        message = yield result

In [29]:
buf = bounded_buffer(2)
print(next(buf))
print(buf.send(Put("A")))
print(buf.send(Put("B")))
print(buf.send(Put("C")))  # full: caller must slow down or retry later
print(buf.send(Get()))
print(buf.send(Put("C")))
print(buf.send(BufferCommand.STATUS))

try:
    buf.send(BufferCommand.STOP)
except StopIteration as exc:
    print("remaining:", exc.value)

BufferResponse(ok=True, status='ready', value=None, size=0, capacity=2)
BufferResponse(ok=True, status='stored', value=None, size=1, capacity=2)
BufferResponse(ok=True, status='stored', value=None, size=2, capacity=2)
BufferResponse(ok=False, status='full', value=None, size=2, capacity=2)
BufferResponse(ok=True, status='retrieved', value='A', size=1, capacity=2)
BufferResponse(ok=True, status='stored', value=None, size=2, capacity=2)
BufferResponse(ok=True, status='status', value=None, size=2, capacity=2)
remaining: ('B', 'C')


In [30]:
buf = bounded_buffer(1)
assert next(buf).status == "ready"
assert buf.send(Put(10)).ok
full = buf.send(Put(20))
assert not full.ok and full.status == "full"
retrieved = buf.send(Get())
assert retrieved.ok and retrieved.value == 10
empty = buf.send(Get())
assert not empty.ok and empty.status == "empty"
try:
    buf.send(BufferCommand.STOP)
except StopIteration as exc:
    assert exc.value == ()
print("Problem 10 checks passed.")

Problem 10 checks passed.


# Problem 11 — Guarantee cleanup with a context manager

Wrap a generator coroutine in a context manager that primes it on entry and closes it on exit, including when the body raises an exception.

### Requirements

- The context manager owns the coroutine lifecycle.
- Cleanup must happen for normal exit and exceptional exit.
- The original exception from the `with` body must propagate.

## Solution 11

In [31]:
@contextmanager
def managed_coroutine(gen: Generator[Any, Any, Any]) -> Iterator[Generator[Any, Any, Any]]:
    try:
        next(gen)
        yield gen
    finally:
        gen.close()


def logging_sink(target: list[Any], cleanup_log: list[str]) -> Generator[None, Any, None]:
    try:
        while True:
            target.append((yield))
    finally:
        cleanup_log.append("closed")

In [32]:
items: list[Any] = []
cleanup: list[str] = []

try:
    with managed_coroutine(logging_sink(items, cleanup)) as sink:
        sink.send("first")
        sink.send("second")
        raise RuntimeError("simulated failure")
except RuntimeError as exc:
    print("body exception propagated:", exc)

print("items:", items)
print("cleanup:", cleanup)

body exception propagated: simulated failure
items: ['first', 'second']
cleanup: ['closed']


In [33]:
items, cleanup = [], []
gen = logging_sink(items, cleanup)
try:
    with managed_coroutine(gen) as sink:
        sink.send(1)
        raise KeyError("boom")
except KeyError:
    pass
assert items == [1]
assert cleanup == ["closed"]
assert getgeneratorstate(gen) == "GEN_CLOSED"
print("Problem 11 checks passed.")

Problem 11 checks passed.


# Problem 12 — Build a reusable coroutine test driver

Coroutine tests often repeat the same pattern: prime, send a sequence, capture responses, optionally stop, and always close.

Implement a helper that:

- accepts a generator and iterable of messages;
- records the first yielded value and all send responses;
- optionally treats one message as terminal and captures `StopIteration.value`;
- closes the generator in `finally` if it is still open.

## Solution 12

In [34]:
@dataclass(frozen=True)
class DriveResult:
    outputs: tuple[Any, ...]
    return_value: Any = None
    terminated: bool = False


def drive_coroutine(
    gen: Generator[Any, Any, Any],
    messages: Iterable[Any],
) -> DriveResult:
    outputs: list[Any] = []
    return_value: Any = None
    terminated = False

    try:
        outputs.append(next(gen))
        for message in messages:
            try:
                outputs.append(gen.send(message))
            except StopIteration as exc:
                return_value = exc.value
                terminated = True
                break
    finally:
        if getgeneratorstate(gen) != "GEN_CLOSED":
            gen.close()

    return DriveResult(tuple(outputs), return_value, terminated)

In [35]:
result = drive_coroutine(
    running_total(),
    [10, 20, TotalCommand.SNAPSHOT, TotalCommand.STOP],
)
print(result)

DriveResult(outputs=(TotalSnapshot(total=0.0, count=0), TotalSnapshot(total=10.0, count=1), TotalSnapshot(total=30.0, count=2), TotalSnapshot(total=30.0, count=2)), return_value=TotalSnapshot(total=30.0, count=2), terminated=True)


In [36]:
result = drive_coroutine(running_total(), [2, 3, TotalCommand.STOP, 999])
assert result.terminated
assert result.return_value == TotalSnapshot(5.0, 2)
assert result.outputs == (
    TotalSnapshot(0.0, 0),
    TotalSnapshot(2.0, 1),
    TotalSnapshot(5.0, 2),
)
print("Problem 12 checks passed.")

Problem 12 checks passed.


# Problem 13 — Capstone: streaming telemetry processor

Build a coroutine that processes readings from many sensors while maintaining independent streaming statistics.

### Data messages

```python
Reading(sensor="cpu", value=73.2, timestamp=1.25)
```

### Control messages

- `ResetSensor(name)` removes one sensor's history.
- `TelemetryCommand.SNAPSHOT` returns all current sensor summaries.
- `TelemetryCommand.STOP` terminates and returns the final summaries.

### Anomaly rule

Before adding a new reading, compute its z-score using the sensor's **existing** sample mean and sample standard deviation. Raise an alert when:

- at least `min_history` prior samples exist;
- prior standard deviation is positive;
- `abs(z_score) >= z_threshold`.

Using prior history avoids letting the candidate reading dilute its own anomaly score.

## Solution 13

In [37]:
@dataclass(frozen=True)
class Reading:
    sensor: str
    value: float
    timestamp: float


@dataclass(frozen=True)
class ResetSensor:
    sensor: str


class TelemetryCommand(Enum):
    SNAPSHOT = auto()
    STOP = auto()


@dataclass
class MutableStats:
    count: int = 0
    mean: float = 0.0
    m2: float = 0.0

    @property
    def sample_variance(self) -> Optional[float]:
        return None if self.count < 2 else self.m2 / (self.count - 1)

    @property
    def sample_stddev(self) -> Optional[float]:
        variance = self.sample_variance
        return None if variance is None else sqrt(variance)

    def update(self, value: float) -> None:
        self.count += 1
        delta = value - self.mean
        self.mean += delta / self.count
        delta2 = value - self.mean
        self.m2 += delta * delta2

    def freeze(self) -> StatsSnapshot:
        return StatsSnapshot(self.count, self.mean, self.m2)


@dataclass(frozen=True)
class Alert:
    sensor: str
    value: float
    timestamp: float
    z_score: float


@dataclass(frozen=True)
class TelemetryResponse:
    status: str
    sensor: Optional[str] = None
    stats: Optional[StatsSnapshot] = None
    alert: Optional[Alert] = None
    all_stats: Optional[dict[str, StatsSnapshot]] = None
    error: Optional[str] = None


def telemetry_processor(
    *,
    z_threshold: float = 3.0,
    min_history: int = 5,
) -> Generator[
    TelemetryResponse,
    Reading | ResetSensor | TelemetryCommand,
    dict[str, StatsSnapshot],
]:
    if z_threshold <= 0:
        raise ValueError("z_threshold must be positive")
    if isinstance(min_history, bool) or not isinstance(min_history, int) or min_history < 2:
        raise ValueError("min_history must be an integer >= 2")

    states: dict[str, MutableStats] = {}

    def freeze_all() -> dict[str, StatsSnapshot]:
        return {name: state.freeze() for name, state in sorted(states.items())}

    message = yield TelemetryResponse(status="ready", all_stats={})

    while True:
        if isinstance(message, Reading):
            if not message.sensor.strip():
                response = TelemetryResponse(status="rejected", error="sensor name is empty")
            elif isinstance(message.value, bool) or not isinstance(message.value, (int, float)):
                response = TelemetryResponse(status="rejected", error="value must be numeric")
            elif not isfinite(float(message.value)) or not isfinite(float(message.timestamp)):
                response = TelemetryResponse(status="rejected", error="value and timestamp must be finite")
            else:
                value = float(message.value)
                state = states.setdefault(message.sensor, MutableStats())
                alert: Optional[Alert] = None
                prior_stddev = state.sample_stddev

                if (
                    state.count >= min_history
                    and prior_stddev is not None
                    and prior_stddev > 0
                ):
                    z_score = (value - state.mean) / prior_stddev
                    if abs(z_score) >= z_threshold:
                        alert = Alert(
                            sensor=message.sensor,
                            value=value,
                            timestamp=float(message.timestamp),
                            z_score=z_score,
                        )

                state.update(value)
                response = TelemetryResponse(
                    status="accepted",
                    sensor=message.sensor,
                    stats=state.freeze(),
                    alert=alert,
                )

        elif isinstance(message, ResetSensor):
            existed = states.pop(message.sensor, None) is not None
            response = TelemetryResponse(
                status="reset" if existed else "not_found",
                sensor=message.sensor,
            )

        elif message is TelemetryCommand.SNAPSHOT:
            response = TelemetryResponse(status="snapshot", all_stats=freeze_all())

        elif message is TelemetryCommand.STOP:
            return freeze_all()

        else:
            response = TelemetryResponse(
                status="rejected",
                error=f"unsupported message: {message!r}",
            )

        message = yield response

In [38]:
processor = telemetry_processor(z_threshold=3.0, min_history=4)
print(next(processor))

baseline = [10.0, 10.5, 9.5, 10.0, 10.2]
for timestamp, value in enumerate(baseline, start=1):
    response = processor.send(Reading("temperature", value, timestamp))
    print(response)

anomaly = processor.send(Reading("temperature", 30.0, 6))
print("anomaly response:", anomaly)
print("snapshot:", processor.send(TelemetryCommand.SNAPSHOT))
print("reset:", processor.send(ResetSensor("temperature")))

try:
    processor.send(TelemetryCommand.STOP)
except StopIteration as exc:
    print("final telemetry state:", exc.value)

TelemetryResponse(status='ready', sensor=None, stats=None, alert=None, all_stats={}, error=None)
TelemetryResponse(status='accepted', sensor='temperature', stats=StatsSnapshot(count=1, mean=10.0, m2=0.0), alert=None, all_stats=None, error=None)
TelemetryResponse(status='accepted', sensor='temperature', stats=StatsSnapshot(count=2, mean=10.25, m2=0.125), alert=None, all_stats=None, error=None)
TelemetryResponse(status='accepted', sensor='temperature', stats=StatsSnapshot(count=3, mean=10.0, m2=0.5), alert=None, all_stats=None, error=None)
TelemetryResponse(status='accepted', sensor='temperature', stats=StatsSnapshot(count=4, mean=10.0, m2=0.5), alert=None, all_stats=None, error=None)
TelemetryResponse(status='accepted', sensor='temperature', stats=StatsSnapshot(count=5, mean=10.04, m2=0.5319999999999999), alert=None, all_stats=None, error=None)
anomaly response: TelemetryResponse(status='accepted', sensor='temperature', stats=StatsSnapshot(count=6, mean=13.366666666666665, m2=332.533333

In [39]:
processor = telemetry_processor(z_threshold=3.0, min_history=4)
ready = next(processor)
assert ready.status == "ready"

for timestamp, value in enumerate([10.0, 10.5, 9.5, 10.0], start=1):
    response = processor.send(Reading("temperature", value, timestamp))
    assert response.status == "accepted"
    assert response.alert is None

anomaly = processor.send(Reading("temperature", 30.0, 5))
assert anomaly.alert is not None
assert anomaly.alert.sensor == "temperature"
assert anomaly.alert.z_score > 3.0

# A second sensor maintains independent state.
processor.send(Reading("pressure", 100.0, 1))
snapshot = processor.send(TelemetryCommand.SNAPSHOT)
assert set(snapshot.all_stats) == {"pressure", "temperature"}
assert snapshot.all_stats["pressure"].count == 1
assert snapshot.all_stats["temperature"].count == 5

rejected = processor.send(Reading("temperature", float("nan"), 6))
assert rejected.status == "rejected"
assert getgeneratorstate(processor) == "GEN_SUSPENDED"

assert processor.send(ResetSensor("pressure")).status == "reset"
try:
    processor.send(TelemetryCommand.STOP)
except StopIteration as exc:
    final_state = exc.value
    assert set(final_state) == {"temperature"}
else:
    raise AssertionError("STOP must terminate")

print("Problem 13 checks passed.")

Problem 13 checks passed.


## Capstone discussion

The telemetry coroutine combines several important techniques:

- an explicit protocol with dataclasses and enums;
- per-key state in a dictionary;
- Welford's online statistics;
- immutable snapshots at the boundary;
- expected validation failures represented as responses;
- terminal state returned through `StopIteration.value`;
- anomaly scoring against prior history.

A production version might add bounded memory, persistence, event-time ordering, duplicate detection, structured logging, and asynchronous I/O.

# Additional mini-problems with compact solutions

These drills reinforce edge cases that frequently cause bugs.

## Mini-problem A — Prove `next(g)` and `g.send(None)` are equivalent when suspended

In [40]:
def counter() -> Generator[int, None, None]:
    value = 0
    while True:
        yield value
        value += 1

c1, c2 = counter(), counter()
assert next(c1) == next(c2) == 0
assert next(c1) == c2.send(None) == 1
assert c1.send(None) == next(c2) == 2
c1.close(); c2.close()
print("Mini-problem A passed.")

Mini-problem A passed.


## Mini-problem B — Capture a generator's explicit return value

In [41]:
def finite_receiver() -> Generator[str, str, tuple[str, ...]]:
    values: list[str] = []
    while len(values) < 3:
        values.append((yield f"need item {len(values) + 1}"))
    return tuple(values)

r = finite_receiver()
print(next(r))
print(r.send("A"))
print(r.send("B"))
try:
    r.send("C")
except StopIteration as exc:
    assert exc.value == ("A", "B", "C")
    print("returned:", exc.value)

need item 1
need item 2
need item 3
returned: ('A', 'B', 'C')


## Mini-problem C — Show that sending into a normal yielding generator may be ignored

In [42]:
def squares(limit: int) -> Generator[int, Any, None]:
    for value in range(limit):
        yield value * value  # the sent value is not assigned anywhere

sq = squares(4)
assert next(sq) == 0
assert sq.send("ignored") == 1
assert sq.send({"also": "ignored"}) == 4
assert next(sq) == 9
try:
    next(sq)
except StopIteration:
    pass
print("Mini-problem C passed.")

Mini-problem C passed.


## Mini-problem D — Demonstrate why yielding during `GeneratorExit` is illegal

In [43]:
def bad_cleanup() -> Generator[str, None, None]:
    try:
        yield "running"
    except GeneratorExit:
        yield "illegal cleanup yield"

bad = bad_cleanup()
assert next(bad) == "running"
try:
    bad.close()
except RuntimeError as exc:
    print("expected RuntimeError:", exc)
finally:
    # The generator may remain suspended after the failed close; close it safely by
    # advancing until termination while suppressing the demonstration artifact.
    try:
        next(bad)
    except StopIteration:
        pass

expected RuntimeError: generator ignored GeneratorExit


## Mini-problem E — Write a small coroutine pipeline

The producer sends raw strings to a normalizer, which forwards normalized strings to a collecting sink.

In [44]:
@autoprime
def normalizer(target: Generator[None, str, None]) -> Generator[None, str, None]:
    while True:
        raw = yield
        normalized = " ".join(raw.strip().lower().split())
        if normalized:
            target.send(normalized)

normalized_items: list[str] = []
collector = list_sink(normalized_items)
normalize = normalizer(collector)

for raw in ["  Hello   WORLD ", "", "  Python  Generators  "]:
    normalize.send(raw)

assert normalized_items == ["hello world", "python generators"]
normalize.close()
collector.close()
print(normalized_items)

['hello world', 'python generators']


# Review questions and answers

### 1. Why can a fresh generator accept only `None`?

Because execution has not yet reached a `yield` expression that can receive a value. Sending `None` is defined as the priming operation.

### 2. What does `send(x)` return?

It returns the **next value yielded** by the generator, not the value `x` itself—unless the generator deliberately yields `x` back.

### 3. What happens when a generator function executes `return result`?

The generator terminates and raises `StopIteration(result)`. The value is available as `exc.value`, and `yield from` captures it automatically.

### 4. What does `close()` do?

It raises `GeneratorExit` at the suspension point. Normal cleanup belongs in `finally`. If the generator yields another value instead of terminating, Python raises `RuntimeError`.

### 5. Why are immutable response objects helpful?

They prevent callers from accidentally mutating internal coroutine state and make tests deterministic.

### 6. When should a class replace a generator coroutine?

Prefer a class when the protocol has many operations, requires introspection, must be serialized, needs multiple independently callable methods, or would be clearer with named methods than with `send()` messages.

# Suggested extension problems — with solution directions

1. **Sliding-window statistics:** maintain a fixed-size `deque`; recompute or maintain removable aggregates. Handle zero variance and window resets.
2. **Rate limiter:** accept `(timestamp, item)` messages and return accepted/rejected responses using a token-bucket state model.
3. **Transactional batcher:** add `BEGIN`, `COMMIT`, and `ROLLBACK` commands; make committed batches immutable.
4. **Hierarchical router:** support topic prefixes such as `system.cpu` and wildcard subscribers.
5. **Checkpointable telemetry:** add a command that returns serializable state and another message that restores validated state.
6. **Coroutine supervisor:** restart a failed child coroutine from a factory, record failure information, and define a retry budget.

A strong solution should specify the protocol first, keep ownership explicit, validate messages, test terminal behavior, and prove cleanup.

# Final takeaways

The difficult part of generator coroutines is not syntax; it is **protocol design**. A robust solution makes lifecycle, message types, return values, error behavior, ownership, and cleanup obvious.

Use these patterns when a compact suspended state machine is genuinely clearer. Otherwise, a class, iterator, queue, or modern asynchronous coroutine may be easier to maintain.